Performing eda on combining both folds of the PanNuke dataset
Dataset link: https://huggingface.co/datasets/RationAI/PanNuke 

In [18]:
!pip install -q datasets huggingface_hub

In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt
from datasets import concatenate_datasets

# Loading the dataset from Hugging Face

In [21]:
from datasets import load_dataset

dataset = load_dataset("RationAI/PanNuke")

In [22]:
# Inspect overall dataset structure
print("Dataset object:")
print(dataset)

print("\nAvailable folds:")
print(dataset.keys())

Dataset object:
DatasetDict({
    fold1: Dataset({
        features: ['image', 'instances', 'categories', 'tissue'],
        num_rows: 2656
    })
    fold2: Dataset({
        features: ['image', 'instances', 'categories', 'tissue'],
        num_rows: 2523
    })
    fold3: Dataset({
        features: ['image', 'instances', 'categories', 'tissue'],
        num_rows: 2722
    })
})

Available folds:
dict_keys(['fold1', 'fold2', 'fold3'])


In [23]:
#length of each fold
print(f"Number of images in  fold1:{len(dataset["fold1"])}" )
print(f"Number of images in  fold2:{len(dataset["fold2"])}")
print(f"Number of images in  fold3:{len(dataset["fold3"])}")


Number of images in  fold1:2656
Number of images in  fold2:2523
Number of images in  fold3:2722


The dataset is divided into 3 folds: fold1, fold2 and fold3. 
Number of images in  fold1:2656
Number of images in  fold2:2523
Number of images in  fold3:2722


Each fold has the images and the corresponding labels, the labels include instance segmentation masks along with categories of nuclei and the tissues

fold 1 and fold 2 will be used for training, total of 5179
fold 3 will be split for validation and testing


# Concatenating all the folds

In [2]:
# Concatenating 2 folds
data = concatenate_datasets([dataset["fold1"], dataset["fold2"], dataset["fold3"]])
#data1= dataset["fold1"]
#data2= dataset["fold2"]
# Print some basic information
print("Number of images in all the folds combined:", len(data))

# Print the feature structure
print("\nDataset features:")
print(data.features)

NameError: name 'dataset' is not defined

# Inspect random image

In [ ]:

# pick a random image index
idx = random.randint(0, len(data)-1)

print("Random sample index:", idx)

# extract the sample
sample = data[idx]

print("\nKeys available in the sample:")
print(sample.keys())

# Inspect pixel values of random image

In [ ]:
# Convert image to numpy array so we can inspect it
img = np.array(sample["image"])

print("Image shape:", img.shape)
print("Image datatype:", img.dtype)

print("\nPixel value range:")
print("Min:", img.min())
print("Max:", img.max())

# Visualise the random image

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5,5))

plt.imshow(img)

plt.title("Original Histopathology Image")

plt.axis("off")

inspect labels of random image

In [ ]:
instances = sample["instances"]
categories = sample["categories"]

print("Number of nuclei in this image:", len(instances))

print("\nCategories for each nucleus:")
print(categories)

print("\nChecking mask-label alignment:")
print("Masks:", len(instances))
print("Labels:", len(categories))

# Visualise the first mask of the random image

In [ ]:
# select first mask
mask0 = np.array(instances[0])

print("Mask shape:", mask0.shape)

print("Unique pixel values in mask:")
print(np.unique(mask0))

In [ ]:
plt.figure(figsize=(4,4))

plt.imshow(mask0, cmap="gray")

plt.title(f"Nucleus 0 | Class {categories[0]}")

plt.axis("off")

In [ ]:
num_masks_to_show =  len(instances)

fig, axes = plt.subplots(1, num_masks_to_show, figsize=(15,4))

for i in range(num_masks_to_show):

    mask = np.array(instances[i])

    axes[i].imshow(mask, cmap="gray")

    axes[i].set_title(f"Nucleus {i}\nClass {categories[i]}")

    axes[i].axis("off")

plt.show()

Drawing contours for each nucleus in the random image

In [ ]:
plt.figure(figsize=(6,6))

# show original image
plt.imshow(img)

print("\nDrawing contours for each nucleus...")

for i, mask in enumerate(instances):

    mask = np.array(mask)

    # draw boundary of the nucleus
    plt.contour(mask, colors="red", linewidths=0.5)

plt.title("Nucleus Masks Overlayed on Image")

plt.axis("off")

# Nuclear size distribution 

In [ ]:
areas = []

print("\nCalculating nucleus sizes...")

for i, mask in enumerate(instances):

    mask = np.array(mask)

    area = mask.sum()   # count white pixels

    areas.append(area)

    print(f"Nucleus {i} area:", area)

In [ ]:
print("\nNucleus Size Statistics")

print("Mean nucleus size:", np.mean(areas))
print("Smallest nucleus:", np.min(areas))
print("Largest nucleus:", np.max(areas))

In [ ]:
num_nuclei = len(instances)

print("\nNuclei in this image:", num_nuclei)

image_area = 256 * 256

density = num_nuclei / image_area

print("Nucleus density:", density)

In [ ]:
# verify masks and labels align

problems = 0

for i in range(100):   # check first 100 images

    sample = data[i]

    if len(sample["instances"]) != len(sample["categories"]):

        print("Mismatch found in image", i)
        problems += 1

print("\nTotal mismatches:", problems)

In [ ]:


# inspect first image
img = np.array(data[0]["image"])

print("Image shape:", img.shape)
print("Datatype:", img.dtype)

print("Pixel range:", img.min(), "to", img.max())

In [ ]:


idx = random.randint(0, len(data)-1)

sample = data[idx]
img = np.array(sample["image"])

print("Random image index:", idx)
print("Tissue type:", sample["tissue"])

plt.figure(figsize=(5,5))
plt.imshow(img)
plt.title("Random Histopathology Image")
plt.axis("off")

In [ ]:
instances = sample["instances"]
categories = sample["categories"]

print("Number of nuclei:", len(instances))

# visualize first few masks
num_show = min(6, len(instances))

fig, axes = plt.subplots(1, num_show, figsize=(15,4))

for i in range(num_show):

    mask = np.array(instances[i])

    axes[i].imshow(mask, cmap="gray")
    axes[i].set_title(f"Nucleus {i}\nClass {categories[i]}")
    axes[i].axis("off")

plt.show()

In [ ]:
plt.figure(figsize=(6,6))

plt.imshow(img)

print("Overlaying masks...")

for mask in instances:

    mask = np.array(mask)

    plt.contour(mask, colors="red", linewidths=0.5)

plt.title("Nuclei Masks Overlay")
plt.axis("off")

# Nuclear class distribution 

In [ ]:
from collections import Counter

class_counts = Counter()

for i in range(len(data)):

    sample = data[i]

    for c in sample["categories"]:

        class_counts[c] += 1

print("Nucleus class distribution:")
print(class_counts)

In [ ]:

import seaborn as sns

# Suppose you've already computed class_counts
# class_counts = Counter({0: 5000, 1: 3000, 2: 2000, 3: 500, 4: 100})

# Convert to two lists for plotting
classes = list(class_counts.keys())
counts = list(class_counts.values())

# Plot
plt.figure(figsize=(8,5))
sns.barplot(x=classes, y=counts,hue=classes,legend=False)
plt.title("Nucleus Class Distribution")
plt.xlabel("Nucleus Class")
plt.ylabel("Number of Nuclei")
plt.show()

In [ ]:
densities = []
'''
densities1=[]
densities2=[]

for i in range (len(data1)):
    sample = data[i]

    densities1.append(len(sample["instances"]))

for i in range (len(data2)):
    sample = data[i]

    densities2.append(len(sample["instances"]))
'''
for i in range(len(data)):

    sample = data[i]

    densities.append(len(sample["instances"]))
'''
print("Average nuclei per image fold1:", np.mean(densities1))
print("Min nuclei  fold1:", np.min(densities1))
print("Max nuclei  fold1:", np.max(densities1))
print("Average nuclei per image in fold2:", np.mean(densities2))
print("Min nuclei in fold2:", np.min(densities2))
print("Max nuclei in fold2:", np.max(densities2))
'''
print("Average nuclei per image in combined fold:", np.mean(densities))
print("Min nuclei in combined fold:", np.min(densities))
print("Max nuclei in combined fold:", np.max(densities))

In [ ]:

'''
plt.figure(figsize=(15, 4))

# Fold 1
plt.subplot(1, 3, 1)
plt.hist(densities1, bins=30)
plt.title("Fold 1")
plt.xlabel("Nuclei per image")
plt.ylabel("Count")

# Fold 2
plt.subplot(1, 3, 2)
plt.hist(densities2, bins=30)
plt.title("Fold 2")
plt.xlabel("Nuclei per image")
'''
# Combined
plt.subplot(1, 3, 3)
plt.hist(densities, bins=30)
plt.title("Combined Fold")
plt.xlabel("Nuclei per image")

plt.tight_layout()
plt.show()

In [ ]:
sizes = []
'''
sizes1 = []
sizes2 = []

for i in range (len(data1)):
    sample = data[i]

    for mask in sample["instances"]:

        mask = np.array(mask)

        sizes1.append(mask.sum())

for i in range (len(data2)):
    sample = data[i]

    for mask in sample["instances"]:

        mask = np.array(mask)

        sizes2.append(mask.sum())
'''
for i in range(len(data)):

    sample = data[i]

    for mask in sample["instances"]:

        mask = np.array(mask)

        sizes.append(mask.sum())

In [ ]:
'''
print("Mean nucleus size for fold1:", np.mean(sizes1))
print("Min nucleus size for fold1:", np.min(sizes1))
print("Max nucleus size for fold1:", np.max(sizes1))

print("Mean nucleus size for fold2:", np.mean(sizes2))
print("Min nucleus size for fold2:", np.min(sizes2))
print("Max nucleus size for fold2:", np.max(sizes2))
'''
print("Mean nucleus size:", np.mean(sizes))
print("Min nucleus size:", np.min(sizes))
print("Max nucleus size:", np.max(sizes))

In [ ]:



'''
# Fold 1
plt.subplot(1, 3, 1)
plt.hist(sizes1, bins=40)
plt.title("Fold 1 - Nucleus Size")
plt.xlabel("Pixels per nucleus")
plt.ylabel("Frequency")

# Fold 2
plt.subplot(1, 3, 2)
plt.hist(sizes2, bins=40)
plt.title("Fold 2 - Nucleus Size")
plt.xlabel("Pixels per nucleus")
'''
# Combined


plt.figure(figsize=(6,4))

plt.hist(sizes, bins=40)

plt.title("Combined - Nucleus Size")
plt.xlabel("Pixels per nucleus")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
colors = []

for i in range(200):   # sample images

    img = np.array(data[i]["image"])

    colors.append(img.reshape(-1,3))
    
colors = np.vstack(colors)

print("Collected color pixels:", colors.shape)

In [ ]:
plt.hist(colors[:,0], bins=50, alpha=0.5, label="Red")
plt.hist(colors[:,1], bins=50, alpha=0.5, label="Green")
plt.hist(colors[:,2], bins=50, alpha=0.5, label="Blue")

plt.title("Pixel Color Distribution")
plt.legend()

The RGB pixel intensity distribution indicates that most pixels lie in the higher range (150–255), suggesting bright images with light backgrounds typical of histopathology slides. The red channel dominates due to eosin staining, while the blue channel reflects nuclei stained by hematoxylin, which is important for segmentation. The green channel is more spread out and less prominent. Overall, the distribution aligns with expected H&E staining patterns and supports effective feature learning after normalization.


# Tissue type distribution 

In [ ]:
tissue_counts = Counter()

for i in range(len(data)):

    tissue_counts[data[i]["tissue"]] += 1

print("Tissue distribution:")
print(tissue_counts)

In [ ]:
# Get tissue label names from dataset metadata
tissue_names = data.features["tissue"].names

print("Tissue Label Mapping:\n")

for i, name in enumerate(tissue_names):
    print(f"{i} → {name}")

In [ ]:
from collections import Counter

tissue_counts = Counter()

for i in range(len(data)):

    tissue_id = data[i]["tissue"]

    tissue_counts[tissue_id] += 1

print("\nTissue Counts (ID → number of images):\n")

for tissue_id, count in sorted(tissue_counts.items()):

    print(f"{tissue_id} ({tissue_names[tissue_id]}) → {count}")

In [ ]:
from collections import Counter

def get_tissue_counts(dataset):
    counts = Counter()
    for i in range(len(dataset)):
        tissue_id = dataset[i]["tissue"]
        counts[tissue_id] += 1
    return counts


counts_fold1 = get_tissue_counts(data1)
counts_fold2 = get_tissue_counts(data2)

# Combined
counts_combined = counts_fold1 + counts_fold2

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Example: counts_fold1, counts_fold2, counts_combined are already defined

# Get all unique tissue IDs from the combined counts
ids = list(counts_combined.keys())
labels = [str(tid) for tid in ids]  # simple labels as strings

plt.figure(figsize=(15, 5))
'''
# Fold 1
plt.subplot(1, 3, 1)
bars = plt.bar(ids, [counts_fold1.get(tid, 0) for tid in ids])
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height, str(height),
             ha='center', va='bottom', fontsize=8)
plt.xticks(ids, labels, rotation=60)
plt.title("Fold 1")
plt.xlabel("Tissue")
plt.ylabel("Count")

# Fold 2
plt.subplot(1, 3, 2)
bars = plt.bar(ids, [counts_fold2.get(tid, 0) for tid in ids])
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height, str(height),
             ha='center', va='bottom', fontsize=8)
plt.xticks(ids, labels, rotation=60)
plt.title("Fold 2")
plt.xlabel("Tissue")
'''
# Combined


plt.figure(figsize=(8,5))

bars = plt.bar(ids, [counts_combined.get(tid, 0) for tid in ids])

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height, str(height),
             ha='center', va='bottom', fontsize=8)

plt.xticks(ids, labels, rotation=60)
plt.title("Combined Tissue Distribution")
plt.xlabel("Tissue")
plt.ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# IDs of tissues
ids = sorted(tissue_counts.keys())

# number of images per tissue
counts = [tissue_counts[i] for i in ids]

# labels showing both ID and tissue name
labels = [f"{i}\n{tissue_names[i]}" for i in ids]


plt.figure(figsize=(12,6))

# create bar plot
bars = plt.bar(ids, counts)


# add value labels on top of each bar
for bar in bars:

    height = bar.get_height()  # y value

    plt.text(
        bar.get_x() + bar.get_width()/2,   # center of bar
        height,                            # height of bar
        str(height),                       # value to print
        ha='center',                       # horizontal alignment
        va='bottom',                       # vertical alignment
        fontsize=10
    )


plt.xticks(ids, labels, rotation=60)

plt.title("Tissue Type Distribution")
plt.xlabel("Tissue ID and Name")
plt.ylabel("Number of Images")

plt.show()

In [ ]:
import numpy as np

nuclei_counts = []

print("Calculating number of nuclei per image...\n")

for i in range(len(data)):

    sample = data[i]

    # number of nuclei = number of masks
    count = len(sample["instances"])

    nuclei_counts.append(count)

print("Finished counting nuclei.")
print("\nTotal images analyzed:", len(nuclei_counts))

print("\nNuclei statistics:")
print("Average nuclei per image:", np.mean(nuclei_counts))
print("Minimum nuclei:", np.min(nuclei_counts))
print("Maximum nuclei:", np.max(nuclei_counts))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.hist(nuclei_counts, bins=30)

plt.title("Distribution of Nuclei per Image")
plt.xlabel("Number of Nuclei")
plt.ylabel("Number of Images")

plt.show()

# Visualising the image with densest nuclei 

In [ ]:
# find index of densest image
dense_index = np.argmax(nuclei_counts)

print("Image with most nuclei:", dense_index)
print("Number of nuclei:", nuclei_counts[dense_index])

In [ ]:
# find the index of the image with the highest nucleus count
dense_index = int(np.argmax(nuclei_counts))

print("Index of densest image:", dense_index)
print("Number of nuclei in this image:", nuclei_counts[dense_index])


# retrieve that image sample
sample = data[dense_index]

# convert image to numpy
img = np.array(sample["image"])

# retrieve nucleus masks
instances = sample["instances"]

print("\nImage shape:", img.shape)
print("Total nuclei in this image:", len(instances))


# visualize the dense cluster
plt.figure(figsize=(6,6))

plt.imshow(img)

print("\nOverlaying nucleus boundaries...")

for i, mask in enumerate(instances):

    mask = np.array(mask)

    # print details for first few nuclei
    if i < 5:
        print(f"Nucleus {i} area:", mask.sum())

    plt.contour(mask, colors="yellow", linewidths=0.5)

plt.title(f"Dense Nucleus Cluster\nTotal nuclei: {len(instances)}")

plt.axis("off")

plt.show()

In [ ]:
import pandas as pd

records = []

print("Building nucleus-level dataset...\n")

for i in range(len(data)):

    sample = data[i]

    tissue_id = sample["tissue"]

    categories = sample["categories"]

    # each category corresponds to one nucleus
    for nucleus_class in categories:

        records.append((tissue_id, nucleus_class))


# convert to dataframe
df = pd.DataFrame(records, columns=["tissue", "nucleus_class"])

print("Total nuclei analyzed:", len(df))

print("\nFirst few rows:")
print(df.head())

In [ ]:
correlation_table = pd.crosstab(df["tissue"], df["nucleus_class"])

print("Tissue vs Nucleus Class Table:\n")

print(correlation_table)

In [ ]:
tissue_names = data.features["tissue"].names

print("\nTissue Label Mapping:\n")

for i, name in enumerate(tissue_names):

    print(f"{i} → {name}")

# **Correlation between tissue type and nucleus class**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(10,6))

plt.imshow(correlation_table)

plt.colorbar(label="Number of Nuclei")

plt.title("Correlation: Tissue Type vs Nucleus Class")

plt.xlabel("Nucleus Class")
plt.ylabel("Tissue Type")

# x-axis labels
plt.xticks(
    range(len(correlation_table.columns)),
    correlation_table.columns
)

# y-axis labels
plt.yticks(
    range(len(correlation_table.index)),
    [tissue_names[i] for i in correlation_table.index]
)

plt.show()

The heatmap shows the distribution of different nucleus classes across various tissue types in the dataset. It reveals a clear imbalance, where certain tissues such as breast and colon contain significantly higher numbers of nuclei, particularly in specific classes, while other tissues and classes have relatively low representation. Some nucleus classes appear sparse or nearly absent across most tissues, indicating potential challenges for model learning due to class imbalance. Overall, the plot highlights both the variability in cellular composition across tissues and the need to account for uneven class distributions during model training.

# **Average nuclear density per tissue**

In [ ]:
density_records = []

for i in range(len(data)):

    sample = data[i]

    tissue = sample["tissue"]

    nuclei_count = len(sample["instances"])

    density_records.append((tissue, nuclei_count))


density_df = pd.DataFrame(
    density_records,
    columns=["tissue", "nuclei_count"]
)

print("First few density records:")
print(density_df.head())

In [ ]:
avg_density = density_df.groupby("tissue")["nuclei_count"].mean()

print("\nAverage nuclei per tissue:\n")
print(avg_density)

In [ ]:
labels = [tissue_names[i] for i in avg_density.index]

plt.figure(figsize=(12,5))

bars = plt.bar(range(len(avg_density)), avg_density)

plt.xticks(range(len(avg_density)), labels, rotation=60)

plt.title("Average Nuclei Density per Tissue")
plt.xlabel("Tissue Type")
plt.ylabel("Average Nuclei per Image")


# print values on top
for bar in bars:

    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f"{height:.2f}",
        ha="center",
        va="bottom"
    )

plt.show()

From the above plot it can be observed that there is uneven nuclei density between the tissue types. The highest nuclear density was observed in the uterine tissue, followed by lung, skin, stomach, kidney.

Note that the number of images for the uterine tissue was the least

# **Average nuclei size per class.**

In [ ]:
size_records = []

for i in range(len(data)):

    sample = data[i]

    masks = sample["instances"]

    classes = sample["categories"]

    for mask, cls in zip(masks, classes):

        mask = np.array(mask)

        size = mask.sum()

        size_records.append((cls, size))


size_df = pd.DataFrame(
    size_records,
    columns=["class", "size"]
)

print("Example nucleus sizes:")
print(size_df.head())

In [ ]:
avg_size = size_df.groupby("class")["size"].mean()

print("\nAverage nucleus size per class:\n")

print(avg_size)

In [ ]:
plt.figure(figsize=(6,4))

bars = plt.bar(avg_size.index, avg_size.values)

plt.title("Average Nucleus Size per Class")

plt.xlabel("Nucleus Class")
plt.ylabel("Average Pixel Area")


# annotate bars
for bar in bars:

    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f"{height:.1f}",
        ha="center",
        va="bottom"
    )

plt.show()

The average nuclei size is different across the different nucleus classes. Highest pixel area was observed in class 0 nuclei, lowest pixel area was observed in class 3 nuclei. 

In [3]:
from datasets import load_dataset
import numpy as np, os, cv2

data = load_dataset("RationAI/PanNuke", split="fold1")

colors = {
    0: [255, 0, 0],
    1: [0, 255, 0],
    2: [0, 0, 255],
    3: [255, 255, 0],
    4: [255, 0, 255],
}

out_dir = "kaggle/working/colored_overlays"
os.makedirs(out_dir, exist_ok=True)

batch_size = 100   # keep smaller for speed
total = len(data)

for start in range(0, total, batch_size):

    end = min(start + batch_size, total)
    print(f"Processing {start} to {end}")

    for i in range(start, end):

        image = np.array(data[i]["image"])
        masks = data[i]["instances"]
        categories = data[i]["categories"]

        color_mask = np.zeros_like(image)

        for mask, cls in zip(masks, categories):
            mask = np.array(mask)

            color = colors.get(cls, [255,255,255])
            color_mask[mask > 0] = color

        overlay = (0.7 * image + 0.3 * color_mask).astype(np.uint8)

        # save instead of plotting (MUCH faster)
        cv2.imwrite(os.path.join(out_dir, f"overlay_{i}.png"), overlay)

README.md: 0.00B [00:00, ?B/s]

data/fold1-00000-of-00001.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

data/fold2-00000-of-00001.parquet:   0%|          | 0.00/264M [00:00<?, ?B/s]

data/fold3-00000-of-00001.parquet:   0%|          | 0.00/289M [00:00<?, ?B/s]

Generating fold1 split:   0%|          | 0/2656 [00:00<?, ? examples/s]

Generating fold2 split:   0%|          | 0/2523 [00:00<?, ? examples/s]

Generating fold3 split:   0%|          | 0/2722 [00:00<?, ? examples/s]

Processing 0 to 100
Processing 100 to 200
Processing 200 to 300
Processing 300 to 400
Processing 400 to 500
Processing 500 to 600
Processing 600 to 700
Processing 700 to 800
Processing 800 to 900
Processing 900 to 1000
Processing 1000 to 1100
Processing 1100 to 1200
Processing 1200 to 1300
Processing 1300 to 1400
Processing 1400 to 1500
Processing 1500 to 1600
Processing 1600 to 1700
Processing 1700 to 1800
Processing 1800 to 1900
Processing 1900 to 2000
Processing 2000 to 2100
Processing 2100 to 2200
Processing 2200 to 2300
Processing 2300 to 2400
Processing 2400 to 2500
Processing 2500 to 2600
Processing 2600 to 2656


In [4]:
import shutil
shutil.make_archive("/kaggle/working/kaggle/working/colored_overlays", "zip", "/kaggle/working/kaggle/working/colored_overlays")

'/kaggle/working/kaggle/working/colored_overlays.zip'